# Import the necessary libraries

In [23]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from scipy import stats

# 1. Đọc dữ liệu
df = pd.read_csv('cleaned_data/VNM_macro_cleaned.csv')
df = df.sort_values('Year').reset_index(drop=True)

# 2. Lấy các biến số kinh tế vĩ mô cốt lõi và loại bỏ giá trị rỗng (NA)
cols = ['Year', 'GDP_Growth_Pct', 'Inflation_CPI_Pct', 'Unemployment_Pct', 'FDI_to_GDP_Pct']
df_clean = df[cols].dropna().copy()

print(f"Số lượng năm quan sát: {len(df_clean)}")

Số lượng năm quan sát: 35


# Giả thuyết 1: Tác động của FDI và Độ mở kinh tế lên Tăng trưởng GDP

In [ ]:
# 1. Đọc dữ liệu từ file CSV
df = pd.read_csv('cleaned_data/VNM_macro_cleaned.csv')

# 2. Lấy các biến số cần thiết cho Giả thuyết 1 và loại bỏ các dòng có giá trị NA
cols = ['Year', 'GDP_Growth_Pct', 'FDI_to_GDP_Pct', 'Economic_Openness_Pct']
df_clean = df[cols].dropna().copy()

# 3. Xây dựng mô hình OLS
# Khai báo các biến độc lập (X) và thêm Hằng số (Constant)
X = df_clean[['FDI_to_GDP_Pct', 'Economic_Openness_Pct']]
X = sm.add_constant(X)

# Khai báo biến phụ thuộc (y)
y = df_clean['GDP_Growth_Pct']

# Fit mô hình hồi quy tuyến tính (OLS)
model_gdp = sm.OLS(y, X).fit()

# 4. In bảng kết quả thống kê (Summary)
print("=== KẾT QUẢ MÔ HÌNH OLS (GIẢ THUYẾT 1) ===")
print(model_gdp.summary())

# 5. Kiểm định chẩn đoán phần dư (Jarque-Bera Test)
jb_stat, jb_p = stats.jarque_bera(model_gdp.resid)
print(f"\n=> Jarque-Bera p-value (Kiểm định phân phối chuẩn của phần dư): {jb_p:.4f}")
if jb_p > 0.05:
    print("Kết luận: Phần dư tuân theo phân phối chuẩn (Đạt yêu cầu).")
else:
    print("Kết luận: Phần dư KHÔNG tuân theo phân phối chuẩn (Vi phạm giả định).")

=== KẾT QUẢ MÔ HÌNH OLS (GIẢ THUYẾT 1) ===
                            OLS Regression Results                            
Dep. Variable:         GDP_Growth_Pct   R-squared:                       0.235
Model:                            OLS   Adj. R-squared:                  0.188
Method:                 Least Squares   F-statistic:                     4.924
Date:                Mon, 18 May 2026   Prob (F-statistic):             0.0137
Time:                        21:41:04   Log-Likelihood:                -59.890
No. Observations:                  35   AIC:                             125.8
Df Residuals:                      32   BIC:                             130.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------

### ĐÁNH GIÁ VÀ GIẢI THÍCH KẾT QUẢ MÔ HÌNH OLS
**Giả thuyết 1:** *"Dòng vốn FDI và Độ mở thương mại thúc đẩy tăng trưởng kinh tế Việt Nam"*

Dựa trên kết quả hồi quy, phương trình mô hình được viết lại như sau:
$$\text{GDP\_Growth} = 6.8467 + 0.2449 \times \text{FDI\_to\_GDP} - 0.0121 \times \text{Economic\_Openness}$$

#### 1. Đánh giá độ phù hợp tổng thể của mô hình
- **Sức mạnh giải thích:** Mô hình giải thích được khoảng **23.5%** sự biến thiên của tốc độ tăng trưởng GDP ($R^2 = 0.235$; Adjusted $R^2 = 0.188$). 
- **Ý nghĩa tổng thể:** Kiểm định F có ý nghĩa thống kê ($Prob(F) = 0.0137 < 0.05$). Điều này khẳng định mô hình tổng thể có giá trị sử dụng và các biến độc lập được chọn có khả năng giải thích chung cho sự thay đổi của biến phụ thuộc.

#### 2. Đánh giá tác động của các hệ số hồi quy
- **Hằng số chặn (const):** Hệ số có giá trị dương ($6.8467$) và có ý nghĩa thống kê rất cao ($p < 0.001$). Về mặt kinh tế, điều này phản ánh mức tăng trưởng nền tảng nội tại. Nếu loại trừ tác động của FDI và độ mở thương mại, nền kinh tế Việt Nam vẫn duy trì động lực tăng trưởng ở mức xấp xỉ 6.85% nhờ các yếu tố nội sinh (tiêu dùng, đầu tư công,...).
- **Tỷ trọng FDI trên GDP (FDI_to_GDP_Pct):** Hệ số mang dấu dương ($+0.2449$) và có ý nghĩa thống kê ở mức 5% ($p = 0.043$). Cụ thể, khi tỷ trọng FDI/GDP tăng 1 điểm %, tốc độ tăng trưởng GDP tăng tương ứng khoảng **0.24 điểm %**. Kết quả này cung cấp bằng chứng thực nghiệm vững chắc chứng minh dòng vốn FDI đóng vai trò tích cực trong việc thúc đẩy tăng trưởng kinh tế.
- **Độ mở kinh tế (Economic_Openness_Pct):** Hệ số mang dấu âm ($-0.0121$) nhưng **không có ý nghĩa thống kê** ($p = 0.108 > 0.10$). Dựa trên tập dữ liệu này, chưa có đủ bằng chứng để khẳng định mức độ mở cửa thương mại có tác động trực tiếp lên tốc độ tăng trưởng GDP. Điều này có thể phản ánh đặc thù gia công xuất khẩu của Việt Nam: kim ngạch XNK lớn nhưng giá trị gia tăng nội địa giữ lại chưa cao.

#### 3. Chẩn đoán khuyết tật mô hình (Phân tích phần dư)
- **Kiểm định phân phối chuẩn:** Kiểm định Jarque-Bera cho $p-value = 0.520 > 0.05$. Không có bằng chứng bác bỏ giả thuyết $H_0$; phần dư của mô hình hoàn toàn tuân theo phân phối chuẩn, đảm bảo tính không chệch của các hệ số ước lượng.
- **Kiểm định tự tương quan:** Chỉ số Durbin-Watson đạt $1.640$, nằm trong vùng an toàn, cho thấy mô hình không gặp vấn đề tự tương quan bậc 1 nghiêm trọng.

> **KẾT LUẬN CHUNG:**
> Dữ liệu thực nghiệm **hỗ trợ một phần Giả thuyết 1**. Dòng vốn FDI có tác động thúc đẩy có ý nghĩa đối với tăng trưởng kinh tế Việt Nam, trong khi mức độ mở cửa thương mại chưa thể hiện được vai trò tác động thống kê một cách rõ ràng trong mô hình tuyến tính ngắn hạn.

# Giả thuyết 2: "Có sự đánh đổi giữa Lạm phát và Thất nghiệp tại Việt Nam" (Đường cong Phillips)

In [25]:
df_clean['Inflation_Lag1'] = df_clean['Inflation_CPI_Pct'].shift(1)

# 2. Xóa bỏ dòng đầu tiên 
df_model = df_clean.dropna().copy()

# 3. Tạo lại biến giả Crisis_Dummy
crisis_years = [1990, 1997, 2008, 2011, 2020] # (Tùy chỉnh danh sách năm)
df_model['Crisis_Dummy'] = df_model['Year'].apply(lambda x: 1 if x in crisis_years else 0)

# 4. Chạy lại OLS với 3 biến: Thất nghiệp, Biến giả, và Lạm phát trễ
X_final = df_model[['Unemployment_Pct', 'Crisis_Dummy', 'Inflation_Lag1']]
X_final = sm.add_constant(X_final)
y_final = df_model['Inflation_CPI_Pct']

model_final = sm.OLS(y_final, X_final).fit()

# In kết quả
print(model_final.summary())

                            OLS Regression Results                            
Dep. Variable:      Inflation_CPI_Pct   R-squared:                       0.487
Model:                            OLS   Adj. R-squared:                  0.436
Method:                 Least Squares   F-statistic:                     9.488
Date:                Mon, 18 May 2026   Prob (F-statistic):           0.000145
Time:                        21:33:51   Log-Likelihood:                -88.906
No. Observations:                  34   AIC:                             185.8
Df Residuals:                      30   BIC:                             191.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                8.5723      3.237  

## Giải thích kết quả OLS
- Mô hình giải thích khoảng 48.7% biến thiên của lạm phát (R-squared = 0.487; Adj. R-squared = 0.436). Kiểm định F có ý nghĩa (Prob(F) = 0.000145), nên mô hình tổng thể phù hợp.
- Hằng số (const) dương và có ý nghĩa (p = 0.013): khi các biến khác bằng 0, mức lạm phát kỳ vọng khoảng 8.57 điểm %.
- Tỷ lệ thất nghiệp có hệ số âm (-2.90) và gần mức ý nghĩa 5% (p = 0.059). Điều này gợi ý thất nghiệp tăng có thể kéo lạm phát giảm, nhưng bằng chứng chưa thật sự chắc chắn ở mức 5%.
- Biến giả khủng hoảng (Crisis_Dummy) dương và rất có ý nghĩa (p = 0.001): trong các năm khủng hoảng, lạm phát cao hơn khoảng 7.26 điểm % so với các năm bình thường.
- Lạm phát trễ 1 năm (Inflation_Lag1) dương và có ý nghĩa (p = 0.046): lạm phát có tính quán tính; lạm phát năm trước tăng 1 điểm % thì lạm phát năm nay tăng khoảng 0.30 điểm %.
- Chẩn đoán phần dư: Durbin-Watson = 1.727 (không cho thấy tự tương quan mạnh); JB p = 0.165, Omnibus p = 0.097 (phần dư gần chuẩn, không có bằng chứng vi phạm nghiêm trọng).